# 00 · Data profiling

Profiles the **structure** of an NSCLC comparator-arm dataset: available
covariates, missingness, endpoint distributions, follow-up, tumour-size
trajectories, and a sanity check against ClinicalTrials.gov benchmarks.

> This notebook runs on **synthetic** data so it works with no gated data.
> To profile the real Project Data Sphere trials, use
> [`02_real_data_profiling.ipynb`](02_real_data_profiling.ipynb) (loads via
> `vca.data_processing.pds_trials`). Everything else is identical because both
> conform to the canonical schema.

In [ ]:
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from vca.data_processing.synthetic import make_synthetic_nsclc

# --- SYNTHETIC stand-in (swap for real PDS data) ---
td = make_synthetic_nsclc(400, seed=42)
# from vca.data_processing.project_data_sphere import load_project_data_sphere
# td = load_project_data_sphere('data/raw/project_data_sphere/<slug>',
#                               'data/raw/column_maps/<slug>.yaml')
td

## Baseline covariates: types, summary, and missingness

In [ ]:
display(td.baseline.dtypes.to_frame('dtype'))
display(td.baseline.describe(include='all').T)
miss = td.baseline.isna().mean().sort_values(ascending=False)
print('missingness (fraction):'); display(miss[miss > 0].to_frame('missing_frac'))

**Flag:** in real PDS comparator arms, biomarkers (EGFR/ALK/PD-L1) and sometimes ECOG are heavily missing — and missing *not at random* (pre-biomarker-era trials). The synthetic generator mimics ~60% PD-L1 missingness. See the ⚠ VALIDITY notes in `docs/methodology.md`.

## Endpoint distributions and follow-up

In [ ]:
e = td.events
print('PFS event rate:', round(e.pfs_event.mean(), 3))
print('OS  event rate:', round(e.os_event.mean(), 3))
print('median follow-up (days, OS):', round(e.os_time_days.median(), 1))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(e.pfs_time_days, bins=30); ax[0].set_title('PFS time (days)')
ax[1].hist(e.os_time_days, bins=30);  ax[1].set_title('OS time (days)')
plt.tight_layout()

In [ ]:
from vca.validation.survival import km_estimate
for ep in ('pfs', 'os'):
    km = km_estimate(e[f'{ep}_time_days'], e[f'{ep}_event'], label=ep)
    plt.step(km.timeline, km.survival, where='post', label=f'{ep.upper()} (median={km.median:.0f}d)')
plt.legend(); plt.ylim(0, 1); plt.xlabel('days'); plt.ylabel('S(t)'); plt.title('Kaplan-Meier')

## Tumour-size (SLD) trajectories

In [ ]:
lg = td.longitudinal
print('measurements / patient:', round(len(lg) / td.n_patients, 2))
for pid, g in list(lg.groupby('patient_id'))[:25]:
    g = g.sort_values('time_days')
    plt.plot(g.time_days, g.sld_mm, alpha=0.4)
plt.xlabel('days'); plt.ylabel('SLD (mm)'); plt.title('Example SLD trajectories (n=25)')

## Sanity check vs ClinicalTrials.gov historical benchmarks

In [ ]:
from pathlib import Path
bpath = Path('data/benchmarks/nsclc_trial_benchmarks.csv')
if bpath.exists():
    b = pd.read_csv(bpath)
    print('Historical median (months) across real NSCLC trial arms:')
    display(b.groupby('endpoint').median_months.describe()[['25%','50%','75%']])
    print('This dataset median PFS (months):', round(e.loc[e.pfs_event==1,'pfs_time_days'].median()/30.44, 2))
    print('This dataset median OS  (months):', round(e.loc[e.os_event==1,'os_time_days'].median()/30.44, 2))
else:
    print('Run `vca-fetch-benchmarks` first to create the benchmark table.')